# Double Inverted Pendulum: Hybrid Swing-Up and LQR Stabilization

This notebook extends the base double-pendulum simulation with a practical hybrid swing-up controller and a local LQR stabilizer near the upright equilibrium.

## System Sketch

The sketch below shows the geometry used in the notebook. Both angles are measured from the upward vertical reference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

theta1_demo = 0.55
theta2_demo = -0.35
l1_demo = 1.0
l2_demo = 0.85

pivot = np.array([0.0, 0.0])
joint1 = np.array([l1_demo * np.sin(theta1_demo), l1_demo * np.cos(theta1_demo)])
joint2 = joint1 + np.array([l2_demo * np.sin(theta2_demo), l2_demo * np.cos(theta2_demo)])

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([pivot[0], joint1[0]], [pivot[1], joint1[1]], 'o-', lw=3, color='tab:blue', label='link 1')
ax.plot([joint1[0], joint2[0]], [joint1[1], joint2[1]], 'o-', lw=3, color='tab:green', label='link 2')
ax.plot([0, 0], [0, 1.25], '--', color='gray', lw=1.5, label='upright reference')

arc1 = Arc(pivot, 0.45, 0.45, angle=0, theta1=90 - np.degrees(theta1_demo), theta2=90, color='tab:blue', lw=2)
arc2 = Arc(joint1, 0.40, 0.40, angle=0, theta1=90, theta2=90 - np.degrees(theta2_demo), color='tab:green', lw=2)
ax.add_patch(arc1)
ax.add_patch(arc2)

ax.text(0.04, 0.28, r'$\theta_1$', color='tab:blue', fontsize=14)
ax.text(joint1[0] + 0.12, joint1[1] + 0.10, r'$\theta_2$', color='tab:green', fontsize=14)
ax.text(joint1[0] / 2 + 0.05, joint1[1] / 2, r'$l_1$', color='tab:blue', fontsize=13)
ax.text((joint1[0] + joint2[0]) / 2 + 0.04, (joint1[1] + joint2[1]) / 2, r'$l_2$', color='tab:green', fontsize=13)
ax.text(-0.08, -0.08, 'pivot', fontsize=11)

ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-0.2, 2.0)
ax.set_aspect('equal', adjustable='box')
ax.set_title('Planar double inverted pendulum geometry')
ax.grid(True, alpha=0.25)
ax.legend(loc='upper right')
plt.show()


## Problem Formulation

We use the state vector

$$
x = \begin{bmatrix} \theta_1 & \theta_2 & \dot{\theta}_1 & \dot{\theta}_2 \end{bmatrix}^T,
$$

where:

- $\theta_1$ is the angle of the first link from the upward vertical.
- $\theta_2$ is the angle of the second link from the upward vertical.
- $u$ is the torque applied at the first joint.

With masses $m_1, m_2$, lengths $l_1, l_2$, and gravity $g$, the nonlinear equations of motion are written here in first-order form:

$$
\dot{x} = f(x,u) =
\begin{bmatrix}
\dot{\theta}_1 \\
\dot{\theta}_2 \\
\ddot{\theta}_1 \\
\ddot{\theta}_2
\end{bmatrix},
$$

with $\Delta = \theta_1 - \theta_2$ and

$$
\ddot{\theta}_1 = \frac{g(2m_1+m_2)\sin\theta_1 + m_2 g \sin(\theta_1-2\theta_2) - 2m_2\sin\Delta\left(l_2\dot{\theta}_2^2 + l_1\dot{\theta}_1^2\cos\Delta\right) + 2u}{l_1\left(2m_1+m_2-m_2\cos(2\Delta)\right)},
$$

$$
\ddot{\theta}_2 = \frac{2\sin\Delta\left(l_1(m_1+m_2)\dot{\theta}_1^2 - g(m_1+m_2)\cos\theta_1 + l_2 m_2 \dot{\theta}_2^2\cos\Delta\right)}{l_2\left(2m_1+m_2-m_2\cos(2\Delta)\right)}.
$$

The upright equilibrium used for stabilization is

$$
x_e = \begin{bmatrix}0 & 0 & 0 & 0\end{bmatrix}^T,
\qquad u_e = 0.
$$

## Linearization Around the Upright Equilibrium

For small deviations $\delta x = x - x_e$ and $\delta u = u - u_e$, the system is approximated by

$$
\delta \dot{x} = A \, \delta x + B \, \delta u,
$$

with Jacobians evaluated at the upright equilibrium:

$$
A = \left.\frac{\partial f}{\partial x}\right|_{(x_e,u_e)}
= \begin{bmatrix}
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1 \\
\frac{g(m_1+m_2)}{l_1 m_1} & -\frac{g m_2}{l_1 m_1} & 0 & 0 \\
-\frac{g(m_1+m_2)}{l_2 m_1} & \frac{g(m_1+m_2)}{l_2 m_1} & 0 & 0
\end{bmatrix},
$$

$$
B = \left.\frac{\partial f}{\partial u}\right|_{(x_e,u_e)}
= \begin{bmatrix}
0 \\
0 \\
\frac{1}{l_1 m_1} \\
0
\end{bmatrix}.
$$

The controllability of $(A,B)$ is checked numerically in the code below.

## LQR Objective

The continuous-time LQR controller minimizes the quadratic cost

$$
J = \int_0^{\infty} \left( \delta x^T Q \, \delta x + \delta u^T R \, \delta u \right) dt,
$$

where $Q \succeq 0$ penalizes state deviation and $R \succ 0$ penalizes control effort. The optimal feedback law is

$$
\delta u = -K \, \delta x,
$$

with

$$
K = R^{-1} B^T P,
$$

and $P$ the stabilizing solution of the continuous algebraic Riccati equation

$$
A^T P + P A - P B R^{-1} B^T P + Q = 0.
$$

The Riccati solution is computed directly from the Hamiltonian matrix using NumPy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display, Markdown
import ipywidgets as widgets

plt.rcParams['figure.figsize'] = (8, 4.5)

DEFAULT_PARAMS = {
    'm1': 1.0,
    'm2': 1.0,
    'l1': 1.0,
    'l2': 1.0,
    'g': 9.81,
}

X_EQ = np.zeros(4)
U_EQ = 0.0


In [ ]:
def dynamics(t, state, params, u=0.0):
    theta1, theta2, dtheta1, dtheta2 = state
    m1 = params['m1']
    m2 = params['m2']
    l1 = params['l1']
    l2 = params['l2']
    g = params['g']

    delta = theta1 - theta2
    denom = 2.0 * m1 + m2 - m2 * np.cos(2.0 * delta)

    ddtheta1 = (
        g * (2.0 * m1 + m2) * np.sin(theta1)
        + m2 * g * np.sin(theta1 - 2.0 * theta2)
        - 2.0 * np.sin(delta) * m2 * (dtheta2**2 * l2 + dtheta1**2 * l1 * np.cos(delta))
        + 2.0 * u
    ) / (l1 * denom)

    ddtheta2 = (
        2.0 * np.sin(delta) * (
            dtheta1**2 * l1 * (m1 + m2)
            - g * (m1 + m2) * np.cos(theta1)
            + dtheta2**2 * l2 * m2 * np.cos(delta)
        )
    ) / (l2 * denom)

    return np.array([dtheta1, dtheta2, ddtheta1, ddtheta2], dtype=float)


def linearized_matrices(params):
    m1 = params['m1']
    m2 = params['m2']
    l1 = params['l1']
    l2 = params['l2']
    g = params['g']

    A = np.array([
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0],
        [g * (m1 + m2) / (l1 * m1), -g * m2 / (l1 * m1), 0.0, 0.0],
        [-g * (m1 + m2) / (l2 * m1), g * (m1 + m2) / (l2 * m1), 0.0, 0.0],
    ])
    B = np.array([[0.0], [0.0], [1.0 / (l1 * m1)], [0.0]])
    return A, B


def controllability_matrix(A, B):
    n = A.shape[0]
    cols = [B]
    for k in range(1, n):
        cols.append(np.linalg.matrix_power(A, k) @ B)
    return np.hstack(cols)


def solve_care(A, B, Q, R):
    R_inv = np.linalg.inv(R)
    H = np.block([
        [A, -B @ R_inv @ B.T],
        [-Q, -A.T],
    ])

    eigvals, eigvecs = np.linalg.eig(H)
    stable = np.where(np.real(eigvals) < 0.0)[0]
    if len(stable) != A.shape[0]:
        raise RuntimeError('Failed to isolate the stable invariant subspace for the CARE.')

    V = eigvecs[:, stable]
    n = A.shape[0]
    U1 = V[:n, :]
    U2 = V[n:, :]
    P = np.real_if_close(U2 @ np.linalg.inv(U1), tol=1000)
    P = np.real(P)
    P = 0.5 * (P + P.T)
    return P.astype(float)


def lqr_gain(params, q_diag, r_scalar):
    A, B = linearized_matrices(params)
    Q = np.diag(q_diag)
    R = np.array([[r_scalar]], dtype=float)
    P = solve_care(A, B, Q, R)
    K = np.linalg.solve(R, B.T @ P)
    return K, P, A, B, Q, R


In [ ]:
def wrap_angle(angle):
    return (angle + np.pi) % (2.0 * np.pi) - np.pi


def angle_error_vector(state):
    return np.array([wrap_angle(state[0]), wrap_angle(state[1])], dtype=float)


def default_hybrid_config():
    return {
        'swing_gain': -2.5,
        'posture_gain': 0.0,
        'damping_gain': 1.2,
        'startup_boost': 6.0,
        'startup_velocity_tol': 0.05,
        'switch_angle': 0.7,
        'switch_velocity': 2.0,
        'capture_angle': 0.6,
        'torque_limit': 12.0,
    }


def total_energy(state, params=None):
    params = DEFAULT_PARAMS if params is None else params
    theta1, theta2, dtheta1, dtheta2 = np.asarray(state, dtype=float)
    m1 = params['m1']
    m2 = params['m2']
    l1 = params['l1']
    l2 = params['l2']
    g = params['g']

    kinetic = (
        0.5 * (m1 + m2) * (l1**2) * dtheta1**2
        + 0.5 * m2 * (l2**2) * dtheta2**2
        + m2 * l1 * l2 * dtheta1 * dtheta2 * np.cos(theta1 - theta2)
    )
    potential = (
        (m1 + m2) * g * l1 * (np.cos(theta1) - 1.0)
        + m2 * g * l2 * (np.cos(theta2) - 1.0)
    )
    return float(kinetic + potential)


def make_lqr_controller(K):
    gain = np.asarray(K, dtype=float)

    def controller(t, x):
        torque = float(-(gain @ (np.asarray(x) - X_EQ))[0])
        return torque, 'lqr'

    return controller


def swing_up_control(state, params=None, config=None):
    params = DEFAULT_PARAMS if params is None else params
    cfg = default_hybrid_config() if config is None else {**default_hybrid_config(), **config}
    theta1, theta2, dtheta1, dtheta2 = np.asarray(state, dtype=float)
    e1 = wrap_angle(theta1)
    e2 = wrap_angle(theta2)
    rel = wrap_angle(theta2 - theta1)
    energy_error = total_energy(state, params=params)

    pump = cfg['swing_gain'] * energy_error * dtheta1
    posture = cfg['posture_gain'] * np.sin(rel) - cfg['damping_gain'] * dtheta1
    capture = -0.6 * np.sin(e1) - 0.2 * np.sin(e2)
    torque = pump + posture

    if np.max(np.abs([e1, e2])) < cfg['capture_angle']:
        torque += capture

    near_downward = np.max(np.abs(angle_error_vector([theta1 - np.pi, theta2 - np.pi, 0.0, 0.0]))) < 0.25
    nearly_rest = abs(dtheta1) + abs(dtheta2) < cfg['startup_velocity_tol']
    if near_downward and nearly_rest:
        torque = cfg['startup_boost']

    torque = float(np.clip(torque, -cfg['torque_limit'], cfg['torque_limit']))
    return torque, 'swing'


def hybrid_control(state, params=None, lqr_controller=None, config=None):
    cfg = default_hybrid_config() if config is None else {**default_hybrid_config(), **config}
    state = np.asarray(state, dtype=float)
    angle_err = angle_error_vector(state)
    velocity_norm = np.linalg.norm(state[2:])
    close_to_upright = np.max(np.abs(angle_err)) < cfg['switch_angle'] and velocity_norm < cfg['switch_velocity']

    if close_to_upright and lqr_controller is not None:
        torque, _ = lqr_controller(0.0, state)
        torque = float(np.clip(torque, -cfg['torque_limit'], cfg['torque_limit']))
        return torque, 'lqr'

    return swing_up_control(state, params=params, config=cfg)


def make_hybrid_controller(K, params=None, config=None):
    params = DEFAULT_PARAMS if params is None else params
    lqr_controller = make_lqr_controller(K)

    def controller(t, x):
        return hybrid_control(x, params=params, lqr_controller=lqr_controller, config=config)

    return controller


def dynamics_controlled(t, state, params=None, controller=None, open_loop_torque=0.0, torque_limit=None):
    params = DEFAULT_PARAMS if params is None else params
    torque = float(open_loop_torque)
    if controller is not None:
        control_out = controller(t, state)
        torque += float(control_out[0] if isinstance(control_out, tuple) else control_out)
    if torque_limit is not None:
        torque = float(np.clip(torque, -torque_limit, torque_limit))
    return dynamics(t, state, params, u=torque)


def rk4_step(f, t, x, dt):
    k1 = f(t, x)
    k2 = f(t + 0.5 * dt, x + 0.5 * dt * k1)
    k3 = f(t + 0.5 * dt, x + 0.5 * dt * k2)
    k4 = f(t + dt, x + dt * k3)
    return x + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


def simulate(x0, params=None, duration=8.0, fps=40, controller=None, open_loop_torque=0.0, torque_limit=None):
    params = DEFAULT_PARAMS if params is None else params
    steps = max(2, int(duration * fps) + 1)
    t = np.linspace(0.0, duration, steps)
    x = np.zeros((4, steps), dtype=float)
    u_hist = np.zeros(steps, dtype=float)
    energy_hist = np.zeros(steps, dtype=float)
    mode_hist = np.empty(steps, dtype=object)
    x[:, 0] = np.asarray(x0, dtype=float)

    def control_law(current_t, current_x):
        torque = float(open_loop_torque)
        mode = 'open-loop'
        if controller is not None:
            result = controller(current_t, current_x)
            if isinstance(result, tuple):
                control_torque, mode = result
            else:
                control_torque, mode = result, 'external'
            torque += float(control_torque)
        if torque_limit is not None:
            torque = float(np.clip(torque, -torque_limit, torque_limit))
        return torque, mode

    for k in range(steps - 1):
        dt = t[k + 1] - t[k]
        torque_k, mode_k = control_law(t[k], x[:, k])
        u_hist[k] = torque_k
        mode_hist[k] = mode_k
        energy_hist[k] = total_energy(x[:, k], params=params)

        def f(current_t, current_x):
            return dynamics_controlled(
                current_t,
                current_x,
                params=params,
                controller=controller,
                open_loop_torque=open_loop_torque,
                torque_limit=torque_limit,
            )

        x[:, k + 1] = rk4_step(f, t[k], x[:, k], dt)

    u_hist[-1], mode_hist[-1] = control_law(t[-1], x[:, -1])
    energy_hist[-1] = total_energy(x[:, -1], params=params)
    return {'t': t, 'y': x, 'u': u_hist, 'energy': energy_hist, 'mode': mode_hist}


def link_positions(sol, params=None):
    params = DEFAULT_PARAMS if params is None else params
    l1 = params['l1']
    l2 = params['l2']
    theta1 = sol['y'][0]
    theta2 = sol['y'][1]
    x1 = l1 * np.sin(theta1)
    y1 = l1 * np.cos(theta1)
    x2 = x1 + l2 * np.sin(theta2)
    y2 = y1 + l2 * np.cos(theta2)
    return x1, y1, x2, y2


## Swing-Up and Hybrid Control

The swing-up phase uses the total mechanical energy

$$
E(x) = T(x) + V(x),
$$

with the upright configuration chosen as the zero-potential reference, so the desired energy is

$$
E_{des} = 0.
$$

The energy-shaping control law used away from the upright region is

$$
u_{swing} = k\,(E - E_{des})\,\dot{\theta}_1,
$$

followed by torque saturation. The exact downward state $[\pi, \pi, 0, 0]$ is a symmetric deadlock for pure energy shaping because $\dot{\theta}_1 = 0$, so the implementation adds a small deterministic startup push when the pendulum is hanging downward and nearly motionless.

The final hybrid controller is

$$
u(x) =
\begin{cases}
u_{LQR}(x), & \text{if } x \text{ is close to the upright equilibrium} \\
u_{swing}(x), & \text{otherwise.}
\end{cases}
$$

This lets the pendulum first gain the required energy and then switch to local LQR stabilization near the unstable upright equilibrium.

### Sources Used

- K. J. Astrom, K. Furuta, M. Iwashiro, T. Hoshino, *Energy Based Strategies for Swinging Up a Double Pendulum*, IFAC World Congress 1999.
- The paper above motivates the staged swing-up idea: energy injection away from upright, then local stabilization near the upright state.
- A practical repository example with the same high-level split between swing-up and balance is: [jitendra825/Inverted-Pendulum-Simulink](https://github.com/jitendra825/Inverted-Pendulum-Simulink).

The notebook implementation below remains a simplified practical hybrid controller, not a line-by-line reproduction of the IFAC paper's exact experimental strategy.

## Controller Construction Check

The next cell computes the controllability rank and prints the default LQR gain used by the interactive tool.

In [ ]:
A_default, B_default = linearized_matrices(DEFAULT_PARAMS)
ctrb = controllability_matrix(A_default, B_default)
rank_ctrb = np.linalg.matrix_rank(ctrb)
K_default, P_default, _, _, Q_default, R_default = lqr_gain(
    DEFAULT_PARAMS,
    q_diag=[120.0, 150.0, 18.0, 20.0],
    r_scalar=0.8,
)

display(Markdown(
    f"**Controllability rank:** {rank_ctrb} / {A_default.shape[0]}  \n"
    f"**Default LQR gain:** `K = {np.array2string(K_default, precision=3)}`"
))


## Downward Start Demo

This example starts from `state = [pi, pi, 0, 0]` and uses the hybrid controller to swing the system up before handing off to LQR stabilization.

In [ ]:
HYBRID_DEFAULTS = default_hybrid_config()
hybrid_controller_default = make_hybrid_controller(K_default, params=DEFAULT_PARAMS, config=HYBRID_DEFAULTS)
downward_state = [np.pi, np.pi, 0.0, 0.0]
hybrid_demo_sol = simulate(
    downward_state,
    params=DEFAULT_PARAMS,
    duration=12.0,
    fps=40,
    controller=hybrid_controller_default,
    torque_limit=HYBRID_DEFAULTS['torque_limit'],
)


## Visualisation Helpers

In [ ]:
def plot_single_solution(sol, title):
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    axes = axes.ravel()

    axes[0].plot(sol['t'], sol['y'][0], label=r'$\theta_1$')
    axes[0].plot(sol['t'], sol['y'][1], label=r'$\theta_2$')
    axes[0].set_title(f'{title}: angles')
    axes[0].set_xlabel('Time [s]')
    axes[0].set_ylabel('Angle [rad]')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(sol['t'], sol['y'][2], label=r'$\dot{\theta}_1$')
    axes[1].plot(sol['t'], sol['y'][3], label=r'$\dot{\theta}_2$')
    axes[1].set_title(f'{title}: angular velocities')
    axes[1].set_xlabel('Time [s]')
    axes[1].set_ylabel('Angular velocity [rad/s]')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].plot(sol['t'], sol['u'], color='tab:red')
    axes[2].set_title(f'{title}: torque')
    axes[2].set_xlabel('Time [s]')
    axes[2].set_ylabel('Torque [Nm]')
    axes[2].grid(True, alpha=0.3)

    axes[3].plot(sol['t'], sol['energy'], color='tab:purple', label='total energy')
    axes[3].axhline(0.0, color='black', ls='--', lw=1, label=r'$E_{des}$')
    axes[3].set_title(f'{title}: energy')
    axes[3].set_xlabel('Time [s]')
    axes[3].set_ylabel('Energy [J]')
    axes[3].grid(True, alpha=0.3)
    axes[3].legend()

    fig.tight_layout()
    return fig


def plot_comparison(solutions):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.ravel()
    styles = {
        'Open-loop': ('tab:orange', '--'),
        'LQR': ('tab:green', '-.'),
        'Hybrid': ('tab:blue', '-'),
    }

    for label, sol in solutions.items():
        color, ls = styles[label]
        axes[0].plot(sol['t'], sol['y'][0], color=color, ls=ls, label=f'{label} $\\theta_1$')
        axes[0].plot(sol['t'], sol['y'][1], color=color, ls=':', label=f'{label} $\\theta_2$')
        axes[1].plot(sol['t'], np.linalg.norm(sol['y'][:2], axis=0), color=color, ls=ls, label=label)
        axes[2].plot(sol['t'], sol['u'], color=color, ls=ls, label=label)
        axes[3].plot(sol['t'], sol['energy'], color=color, ls=ls, label=label)

    axes[0].set_title('Angles comparison')
    axes[0].set_xlabel('Time [s]')
    axes[0].set_ylabel('Angle [rad]')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=9, ncol=2)

    axes[1].set_title('Distance from upright')
    axes[1].set_xlabel('Time [s]')
    axes[1].set_ylabel(r'$\|[\theta_1,\theta_2]\|_2$')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    axes[2].set_title('Control effort comparison')
    axes[2].set_xlabel('Time [s]')
    axes[2].set_ylabel('Torque [Nm]')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()

    axes[3].axhline(0.0, color='black', ls='--', lw=1, label=r'$E_{des}$')
    axes[3].set_title('Energy comparison')
    axes[3].set_xlabel('Time [s]')
    axes[3].set_ylabel('Energy [J]')
    axes[3].grid(True, alpha=0.3)
    axes[3].legend()

    fig.tight_layout()
    return fig


def animate_solution(sol, params=None, title='Simulation', interval=40, color='tab:blue', max_frames=180):
    params = DEFAULT_PARAMS if params is None else params
    x1, y1, x2, y2 = link_positions(sol, params)
    reach = params['l1'] + params['l2']
    stride = max(1, len(sol['t']) // max_frames)
    frame_idx = np.arange(0, len(sol['t']), stride, dtype=int)

    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    ax.set_xlim(-reach - 0.2, reach + 0.2)
    ax.set_ylim(-reach - 0.2, reach + 0.2)
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.25)
    ax.set_title(title)

    rod, = ax.plot([], [], 'o-', lw=3, color=color)
    trail, = ax.plot([], [], '-', lw=1.5, alpha=0.45, color=color)
    time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes)

    def update(frame_number):
        frame = frame_idx[frame_number]
        rod.set_data([0.0, x1[frame], x2[frame]], [0.0, y1[frame], y2[frame]])
        start = max(0, frame_number - 25)
        tail = frame_idx[start:frame_number + 1]
        trail.set_data(x2[tail], y2[tail])
        time_text.set_text(f"t = {sol['t'][frame]:.2f} s")
        return rod, trail, time_text

    ani = FuncAnimation(fig, update, frames=len(frame_idx), interval=interval, blit=True)
    html = HTML(ani.to_jshtml())
    plt.close(fig)
    return html


## Interactive Dashboard

The interactive tool below focuses only on the hybrid swing-up controller so the notebook remains responsive. Free response and pure local LQR are left out of the interactive path for speed.

In [ ]:
output = widgets.Output()

theta1_slider = widgets.FloatSlider(value=np.pi, min=-np.pi, max=np.pi, step=0.01, description='theta1')
theta2_slider = widgets.FloatSlider(value=np.pi, min=-np.pi, max=np.pi, step=0.01, description='theta2')
dtheta1_slider = widgets.FloatSlider(value=0.0, min=-6.0, max=6.0, step=0.05, description='dtheta1')
dtheta2_slider = widgets.FloatSlider(value=0.0, min=-6.0, max=6.0, step=0.05, description='dtheta2')
duration_slider = widgets.FloatSlider(value=10.0, min=2.0, max=20.0, step=0.5, description='duration')
open_loop_torque_slider = widgets.FloatSlider(value=0.0, min=-8.0, max=8.0, step=0.1, description='bias torque')
torque_limit_slider = widgets.FloatSlider(value=12.0, min=1.0, max=40.0, step=0.5, description='limit')
switch_angle_slider = widgets.FloatSlider(value=0.7, min=0.08, max=1.5, step=0.01, description='switch ang')
switch_velocity_slider = widgets.FloatSlider(value=2.0, min=0.1, max=6.0, step=0.05, description='switch vel')
capture_angle_slider = widgets.FloatSlider(value=0.6, min=0.1, max=1.5, step=0.01, description='capture')
swing_gain_slider = widgets.FloatSlider(value=-2.5, min=-20.0, max=-0.1, step=0.1, description='swing gain')
posture_gain_slider = widgets.FloatSlider(value=0.0, min=0.0, max=10.0, step=0.1, description='posture')
damping_gain_slider = widgets.FloatSlider(value=1.2, min=0.0, max=6.0, step=0.05, description='damping')
startup_boost_slider = widgets.FloatSlider(value=6.0, min=0.0, max=12.0, step=0.1, description='startup')
q1_slider = widgets.FloatLogSlider(value=80.0, base=10, min=0.0, max=3.4, step=0.01, description='Q theta1')
q2_slider = widgets.FloatLogSlider(value=300.0, base=10, min=0.0, max=3.4, step=0.01, description='Q theta2')
q3_slider = widgets.FloatLogSlider(value=80.0, base=10, min=-1.0, max=3.0, step=0.01, description='Q dtheta1')
q4_slider = widgets.FloatLogSlider(value=80.0, base=10, min=-1.0, max=3.0, step=0.01, description='Q dtheta2')
r_slider = widgets.FloatLogSlider(value=0.6, base=10, min=-3.0, max=2.0, step=0.01, description='R')
show_animation_checkbox = widgets.Checkbox(value=True, description='show animation')


def render(theta1, theta2, dtheta1, dtheta2, duration, open_loop_torque, limit, switch_angle, switch_velocity, capture_angle, swing_gain, posture_gain, damping_gain, startup, q1, q2, q3, q4, r, show_animation):
    with output:
        output.clear_output(wait=True)

        q_diag = [q1, q2, q3, q4]
        K, P, A, B, Q, R = lqr_gain(DEFAULT_PARAMS, q_diag=q_diag, r_scalar=r)
        lqr_controller = make_lqr_controller(K)
        hybrid_cfg = {
            'swing_gain': swing_gain,
            'posture_gain': posture_gain,
            'damping_gain': damping_gain,
            'startup_boost': startup,
            'switch_angle': switch_angle,
            'switch_velocity': switch_velocity,
            'capture_angle': capture_angle,
            'torque_limit': limit,
        }
        hybrid_controller = make_hybrid_controller(K, params=DEFAULT_PARAMS, config=hybrid_cfg)
        x0 = [theta1, theta2, dtheta1, dtheta2]

        hybrid_sol = simulate(
            x0,
            params=DEFAULT_PARAMS,
            duration=duration,
            fps=24,
            controller=hybrid_controller,
            open_loop_torque=open_loop_torque,
            torque_limit=limit,
        )

        eigvals = np.linalg.eigvals(A - B @ K)
        lqr_steps = int(np.sum(np.array(hybrid_sol['mode']) == 'lqr'))
        display(Markdown(
            f"**LQR gain** `K = {np.array2string(K, precision=3)}`  \\n"
            f"**Closed-loop eigenvalues** `{np.array2string(eigvals, precision=3)}`  \\n"
            f"**Hybrid controller** used LQR on `{lqr_steps}` out of `{len(hybrid_sol['mode'])}` recorded steps.  \\n"
            f"**Final angle norm** `{np.linalg.norm(angle_error_vector(hybrid_sol['y'][:, -1])):.3f}`"
        ))

        fig = plot_single_solution(hybrid_sol, 'Hybrid swing-up + LQR')
        display(fig)
        plt.close(fig)
        if show_animation:
            display(animate_solution(hybrid_sol, title='Hybrid swing-up animation', color='tab:blue'))


controls = {
    'theta1': theta1_slider,
    'theta2': theta2_slider,
    'dtheta1': dtheta1_slider,
    'dtheta2': dtheta2_slider,
    'duration': duration_slider,
    'open_loop_torque': open_loop_torque_slider,
    'limit': torque_limit_slider,
    'switch_angle': switch_angle_slider,
    'switch_velocity': switch_velocity_slider,
    'capture_angle': capture_angle_slider,
    'swing_gain': swing_gain_slider,
    'posture_gain': posture_gain_slider,
    'damping_gain': damping_gain_slider,
    'startup': startup_boost_slider,
    'q1': q1_slider,
    'q2': q2_slider,
    'q3': q3_slider,
    'q4': q4_slider,
    'r': r_slider,
    'show_animation': show_animation_checkbox,
}

ui = widgets.VBox([
    widgets.HTML('<b>Hybrid swing-up controller</b>'),
    widgets.HTML('<b>Initial condition</b>'),
    theta1_slider,
    theta2_slider,
    dtheta1_slider,
    dtheta2_slider,
    duration_slider,
    open_loop_torque_slider,
    torque_limit_slider,
    widgets.HTML('<b>Hybrid switching and swing-up parameters</b>'),
    switch_angle_slider,
    switch_velocity_slider,
    capture_angle_slider,
    swing_gain_slider,
    posture_gain_slider,
    damping_gain_slider,
    startup_boost_slider,
    widgets.HTML('<b>LQR weights</b>'),
    q1_slider,
    q2_slider,
    q3_slider,
    q4_slider,
    r_slider,
    show_animation_checkbox,
])

interactive = widgets.interactive_output(render, controls)
display(ui, output, interactive)
